# Cal3_S2Stereo

A `Cal3_S2Stereo` represents the calibration for a stereo camera rig. It assumes that both cameras in the rig share the same intrinsic calibration parameters but are separated by a horizontal baseline. 

This class inherits from the standard 5-parameter `Cal3_S2` model and adds a sixth parameter, the baseline `b`. The six parameters are, therefore: two focal lengths ($f_x, f_y$), skew ($s$), the principal point ($u_0, v_0$), and the baseline ($b$). The calibration matrix $K$ is identical to that of `Cal3_S2` and represents the intrinsics of the *left* camera:

$$
K = \begin{bmatrix} f_x & s & u_0 \\ 0 & f_y & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$

The `uncalibrate` and `calibrate` methods of this class behave identically to those in `Cal3_S2`, converting points for a single camera. The baseline parameter `b` is primarily used by other classes like `StereoCamera` to compute the disparity between the left and right images during 3D projection.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/Cal3_S2Stereo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam-develop

In [2]:
import gtsam
import numpy as np
from gtsam import Cal3_S2Stereo, Point2, Point3, Pose3, StereoCamera

## Initialization

A `Cal3_S2Stereo` object can be initialized in several ways:

In [3]:
# Default constructor
cal_default = Cal3_S2Stereo()
print(f"Default constructor:\n{cal_default}\n")

# From individual parameters: fx, fy, s, u0, v0, b
fx, fy, s, u0, v0, b = 1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5
cal_params = Cal3_S2Stereo(fx, fy, s, u0, v0, b)
print(f"From parameters (fx, fy, s, u0, v0, b):\n{cal_params}\n")

# From a 6-vector [fx, fy, s, u0, v0, b]
param_vector = np.array([1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5])
cal_vector = Cal3_S2Stereo(param_vector)
print(f"From a 6-vector [fx, fy, s, u0, v0, b]:\n{cal_vector}\n")

# From fov in degrees, image dimensions, and stereo baseline
fov, w, h, b = 100, 600, 500, 0.5
cal_fov = Cal3_S2Stereo(fov, w, h, b)
print(f"From fov, image dimensions, and baseline:\n{cal_fov}\n")

Default constructor:
K: 1 0 0
0 1 0
0 0 1
Baseline: 1


From parameters (fx, fy, s, u0, v0, b):
K: 1500  0.1  320
   0 1600  240
   0    0    1
Baseline: 0.5


From a 6-vector [fx, fy, s, u0, v0, b]:
K: 1500  0.1  320
   0 1600  240
   0    0    1
Baseline: 0.5




TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. gtsam.gtsam.Cal3_S2Stereo()
    2. gtsam.gtsam.Cal3_S2Stereo(fx: float, fy: float, s: float, u0: float, v0: float, b: float)
    3. gtsam.gtsam.Cal3_S2Stereo(v: numpy.ndarray[numpy.float64[m, 1]])

Invoked with: 100, 600, 500, 0.5

## Properties

Since `Cal3_S2Stereo` inherits from `Cal3_S2`, it has access to all of its parent's property methods, in addition to its own specific methods, as follows:
- `baseline()`: Returns the horizontal baseline $b$.

In [16]:
fx, fy, s, u0, v0, b = 1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5
cal = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

print("Calibration object properties:")
print(f"fx: {cal.fx()}")
print(f"fy: {cal.fy()}")
print(f"skew: {cal.skew()}")
print(f"u0: {cal.px()}")
print(f"v0: {cal.py()}")
print(f"Baseline: {cal.baseline()}")
print(f"Principal Point: {cal.principalPoint()}")
print(f"Vector [fx, fy, s, u0, v0, b]: {cal.vector()}")
print(f"Aspect ratio: {cal.aspectRatio()}")
print(f"K matrix:\n{cal.K()}")

Calibration object properties:
fx: 1500.0
fy: 1600.0
skew: 0.1
u0: 320.0
v0: 240.0
Baseline: 0.5
Principal Point: [320. 240.]
Vector [fx, fy, s, u0, v0, b]: [1.5e+03 1.6e+03 1.0e-01 3.2e+02 2.4e+02 5.0e-01]
Aspect ratio: 0.9375
K matrix:
[[1.5e+03 1.0e-01 3.2e+02]
 [0.0e+00 1.6e+03 2.4e+02]
 [0.0e+00 0.0e+00 1.0e+00]]


## Basic Operations

The `calibrate` and `uncalibrate` methods work identically to their `Cal3_S2` counterparts, converting points for a single camera's perspective. The baseline does not affect these calculations directly but is used in higher-level classes like `StereoCamera`.

### `calibrate()`

The `calibrate(p)` method converts a 2D point `p` from image pixel coordinates $(u, v)$ to normalized image coordinates $(x, y)$.

In [18]:
fx, fy, s, u0, v0, b = 1000.0, 1000.0, 0.0, 320.0, 240.0, 0.5
cal = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

p_pixels = Point2(420, 290)
p_norm = cal.calibrate(p_pixels)
print(f"Pixel point: {p_pixels}")
print(f"Calibrated (normalized) point: {p_norm}")

# This method can optionally compute Jacobians.
Dcal_calibrate = np.zeros((2, 6), order='F') # Note shape is 2x6
Dp_calibrate = np.zeros((2, 2), order='F')
_ = cal.calibrate(p_pixels, Dcal_calibrate, Dp_calibrate)
print(f"\nJacobian Dcal_calibrate:\n{Dcal_calibrate}")
print(f"\nJacobian Dp_calibrate:\n{Dp_calibrate}")

AttributeError: 'gtsam.gtsam.Cal3_S2Stereo' object has no attribute 'calibrate'

### `uncalibrate()`

The `uncalibrate(p)` method converts a 2D point `p` from normalized coordinates back to pixel coordinates.

In [19]:
p_pixels_recovered = cal.uncalibrate(p_norm)
print(f"Normalized point: {p_norm}")
print(f"Uncalibrated (pixel) point: {p_pixels_recovered}")

# This method can also optionally compute Jacobians.
Dcal_uncalibrate = np.zeros((2, 6), order='F')
Dp_uncalibrate = np.zeros((2, 2), order='F')
_ = cal.uncalibrate(p_norm, Dcal_uncalibrate, Dp_uncalibrate)
print(f"\nJacobian Dcal_uncalibrate:\n{Dcal_uncalibrate}")
print(f"\nJacobian Dp_uncalibrate:\n{Dp_uncalibrate}")

AttributeError: 'gtsam.gtsam.Cal3_S2Stereo' object has no attribute 'uncalibrate'

## Manifold Operations

`Cal3_S2Stereo` is a 6-dimensional manifold and supports `retract` and `localCoordinates` to enable optimization. These operations apply updates in the 6-dimensional tangent space.

In [20]:
print("Original cal_model:")
print(cal)

# Retract: Apply a delta to the calibration parameters
delta_vec = np.array([10.0, 20.0, 0.05, 1.0, -1.0, 0.01])
cal_retracted = cal.retract(delta_vec)
print(f"\nDelta vector: {delta_vec}")
print("\nRetracted cal_retracted:")
print(cal_retracted)

# Local coordinates: Find the delta between two calibrations
local_coords = cal.localCoordinates(cal_retracted)
print("\nLocal coordinates from cal_model to cal_retracted:")
print(local_coords)

Original cal_model:
K: 1000    0  320
   0 1000  240
   0    0    1
Baseline: 0.5


Delta vector: [ 1.e+01  2.e+01  5.e-02  1.e+00 -1.e+00  1.e-02]

Retracted cal_retracted:
K: 1010 0.05  321
   0 1020  239
   0    0    1
Baseline: 0.51


Local coordinates from cal_model to cal_retracted:
[ 1.e+01  2.e+01  5.e-02  1.e+00 -1.e+00  1.e-02]


## Relationship with `StereoCamera`

The `Cal3_S2Stereo` object is most powerful when used with the `StereoCamera` class. A `StereoCamera` combines a `Pose3` (the pose of the left camera) and a `Cal3_S2Stereo` calibration to project 3D points into the stereo image plane, producing a `StereoPoint2` which contains the left and right image coordinates ($u_L, u_R, v$).

In [21]:
fx, fy, s, u0, v0, b = 1000, 1000, 0, 640, 360, 0.5
stereo_calibration = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

# Define the pose of the stereo camera rig in the world frame (left camera's pose)
camera_pose = Pose3(gtsam.Rot3.Ypr(-np.pi/2, 0, -np.pi/2), Point3(0, 0, 5.0))

# Create a StereoCamera object
stereo_camera = StereoCamera(camera_pose, stereo_calibration)

# Define a 3D point in the world frame
p_world = Point3(5, 3, 2)

# Project the 3D point into the stereo camera
p_stereo = stereo_camera.project(p_world)

print(f"3D Point in world frame: {p_world}")
print(f"StereoCamera pose:\n{stereo_camera.pose()}")
print(f"Stereo calibration:\n{stereo_camera.calibration()}")
print(f"\nProjected StereoPoint2 (u_L, u_R, v): {p_stereo}")


3D Point in world frame: [5. 3. 2.]
StereoCamera pose:
R: [
	6.12323e-17, 6.12323e-17, 1;
	-1, 3.7494e-33, 6.12323e-17;
	-0, -1, 6.12323e-17
]
t: 0 0 5

Stereo calibration:
K: 1000    0  640
   0 1000  360
   0    0    1
Baseline: 0.5


Projected StereoPoint2 (u_L, u_R, v): (40, -60, 960)



## Additional Resources

The following resources provide more detailed explanations of camera calibration and stereo vision.

### Video(s)
- ["Simple Stereo | Camera Calibration"](https://youtu.be/hUVyDabn1Mg?si=3zIrPPML-H7Zu5_i) by Dr. Shree Nayar from Columbia University

## Source
- [Cal3_S2Stereo.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3_S2Stereo.h)
- [Cal3_S2Stereo.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3_S2Stereo.cpp)